In [16]:
from rapidsegment import StrategicSegmentBuilder, StrategicSegmentScore, UniversalDataLoader

# Example Feature Groupings Setup
variable_groups = {
    "Demographics": [
        "age", 
        "gender", 
        "education_level", 
        "dependents", 
        "home_ownership", 
        "city_tier"
    ],
    "Employment": [
        "employment_type", 
        "years_employed"
    ],
    "Financial_Profile": [
        "annual_income", 
        "credit_score", 
        "existing_debt", 
        "account_balance", 
        "monthly_spend", 
        "loan_to_income_ratio", 
        "previous_defaults", 
        "number_of_cards"
    ],
    "Application_Details": [
        "requested_limit", 
        "application_channel"
    ]
}

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000,5000, 1000, 500, 100],  # Example values for min_sample_size   
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="is_approved",
    # n_jobs=-1,
    min_sample_size=1000,
    min_lift=1.1,
    min_events = 5,
    top_n_vars=15,
    max_segments=10,
    enable_diversity=True,
    max_feature_reuse=1,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=variable_groups,
    ignore_features=['application_id'],
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/RapidSegment/credit_app_data.parquet").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-12 15:25:24,563 | INFO     | [builder.py:337] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB


2026-07-12 15:25:25,560 | INFO     | [builder.py:133] | Feature group validation passed. (18 features mapped)
2026-07-12 15:25:25,561 | INFO     | [builder.py:352] | Dynamic Grid Search Enabled: 10 total configurations.
2026-07-12 15:25:25,563 | INFO     | [builder.py:373] | Iteration 1 | Remaining Volume: 500,000 | Base Rate: 26.77%
2026-07-12 15:25:33,216 | INFO     | [builder.py:509] | Feature Usage Tracker Update -> 'loan_to_income_ratio' used count = 1
2026-07-12 15:25:33,216 | INFO     | [builder.py:524] | Segment 1 Captured (Size Floor: 10000 | Lift Floor: 2.0): loan_to_income_ratio < 0.04
2026-07-12 15:25:34,564 | INFO     | [builder.py:373] | Iteration 2 | Remaining Volume: 475,531 | Base Rate: 24.44%
2026-07-12 15:25:42,676 | INFO     | [builder.py:509] | Feature Usage Tracker Update -> 'credit_score' used count = 1
2026-07-12 15:25:42,676 | INFO     | [builder.py:524] | Segment 2 Captured (Size Floor: 10000 | Lift Floor: 2.0): credit_score >= 821.50
2026-07-12 15:25:43,845 |

In [17]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate   |        lift        |
+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+
|   0.0   |   451198.0  |    102244.0   | 22.66056143865886  |      26.7676       |      90.2396      | 0.8465667986169421 |
|   1.0   |   24469.0   |    17622.0    | 72.01765499203073  |      26.7676       | 4.893800000000001 | 2.690478600697512  |
|   2.0   |   24333.0   |    13972.0    | 57.419964657050095 |      26.7676       |       4.8666      | 2.145129360011734  |
+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+


In [18]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   loan_to_income_ratio=(-inf, 0.04)
SQL Filter: loan_to_income_ratio < 0.04
--------------------------------------------------
Segment ID: 2
Raw Rule:   credit_score=[821.50, inf)
SQL Filter: credit_score >= 821.50
--------------------------------------------------


In [19]:
builder.explain_feature_journey("city_tier")

AUDIT TRAIL FOR FEATURE: 'city_tier'

[Iteration 1]
  • Current dynamic IV   : 0.0023
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : loan_to_income_ratio=(-inf, 0.04) (Variables: ['loan_to_income_ratio'])

[Iteration 2]
  • Current dynamic IV   : 0.0025
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : credit_score=[821.50, inf) (Variables: ['credit_score'])

[Iteration 3]
  • Current dynamic IV   : 0.0018
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search


In [20]:
# Preparing the dataset for scoring and decile banding.
import duckdb
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
            SELECT *, 
            CASE WHEN loan_to_income_ratio < 0.07
            THEN 1 ELSE 0 END AS seg_1,
            FROM predicted
""").df()
conn.close()

In [21]:
scorer = StrategicSegmentScore(
    target_col="is_approved",
    primary_key="application_id",
    segment_cols=["seg_1"],
)

In [22]:
scorer.calculate_and_export_weights(predicted)

2026-07-12 15:25:53,244 | INFO     | [scorer.py:50] | Initializing out-of-core DuckDB scorecard engine...
2026-07-12 15:25:54,736 | INFO     | [scorer.py:88] | Computing harmonic scorecard weights...
2026-07-12 15:25:54,737 | INFO     | [scorer.py:117] | Scoring 100M rows natively via C++ database engine...
2026-07-12 15:25:55,020 | INFO     | [scorer.py:132] | Dataset Zero-Inflation Rate: 73.23%
2026-07-12 15:25:55,021 | INFO     | [scorer.py:135] | Calibrating deciles across active populations...


{'model_metadata': {'total_training_population': 500000,
  'active_scored_population': 500000,
  'active_population_pct': 100.0,
  'baseline_event_rate': 0.2677},
 'segment_weights': {'seg_1': {'weight': 88,
   'lift': 2.6061,
   'response_rate': 0.6976,
   'capture_rate': 0.2241}},
 'decile_min_thresholds': {'1': 88,
  '2': 0,
  '3': 0,
  '4': 0,
  '5': 0,
  '6': 0,
  '7': 0,
  '8': 0,
  '9': 0,
  '10': 0}}

In [23]:
import logging, json
scorecard_json_path = "/workspaces/RapidSegment/scored_experiment_2026_07_12_15_22_08.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-07-12 15:25:55,106 | INFO     | [1697935397.py:3] | Loading scorecard model artifact from /workspaces/RapidSegment/scored_experiment_2026_07_12_15_22_08.json...


In [24]:
model_artifact.get("decile_min_thresholds")

{'1': 88,
 '2': 0,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [25]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 88


In [26]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 88 ELSE 0 END AS seg_1_weighted,
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted ) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=88 THEN 1
                    WHEN total_weight >= 0 THEN 2
                    WHEN total_weight >= 0 THEN 3
                    WHEN total_weight >= 0 THEN 4
                    WHEN total_weight >= 0 THEN 5
                    WHEN total_weight >= 0 THEN 6
                    WHEN total_weight >= 0 THEN 7
                    WHEN total_weight >= 0 THEN 8
                    WHEN total_weight >= 0 THEN 9
                    WHEN total_weight >= 0 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [27]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(is_approved) AS events, 
                    (SUM(is_approved)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()
scored

,decile_band,count,events,response_rate
0,1,42996,29994.0,69.759978
1,2,457004,103844.0,22.722777
